# File Tracking - Read-Before-Write Pattern

This notebook demonstrates how FlowBook tracks "read-before-write" file dependencies.

**Key concept**: Only files that are read *before* being written in a cell
are tracked as reads. This is important for reproducibility - if a cell
reads a file that another cell writes, we need to track that dependency.

In [ ]:
import json

## Cell A: Write initial config file

This cell writes `config.json`. The file will appear in `writes`.

In [ ]:
# Write initial configuration
config = {
    'version': 1,
    'debug': False,
    'max_retries': 3
}

with open('config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Created config.json')

## Cell B: Read config and use it

This cell reads `config.json`. The file will appear in `reads` as `file:config.json`.
If Cell A is re-run, Cell B becomes stale because its file dependency changed.

In [ ]:
# Read the config and use it
with open('config.json', 'r') as f:
    loaded_config = json.load(f)

print(f"Version: {loaded_config['version']}")
print(f"Debug mode: {loaded_config['debug']}")
print(f"Max retries: {loaded_config['max_retries']}")

## Cell C: Read-then-write in same cell

When a file is read and then written in the *same* cell, it's tracked as read-before-write.
This is a common pattern for updating files.

In [ ]:
# Read config, modify, and write back
with open('config.json', 'r') as f:
    config = json.load(f)

# Update a value
config['version'] = 2
config['new_feature'] = True

with open('config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Updated config.json to version 2')

## Backward mutation detection

If Cell C (which writes config.json) is run *after* Cell B (which reads it),
FlowBook detects this as a potential reproducibility violation - the file
that Cell B depends on was modified by a later cell.

In [ ]:
# Verify the updated config
with open('config.json', 'r') as f:
    final_config = json.load(f)

print('Final config:')
print(json.dumps(final_config, indent=2))

## Cleanup

In [ ]:
import os
if os.path.exists('config.json'):
    os.remove('config.json')
    print('Cleaned up config.json')